In [7]:
#r "nuget: ScottPlot, 5.1.59"
#r "nuget: SkiaSharp.NativeAssets.Linux.NoDependencies, 3.119.2"
#r "../task14/bin/Debug/net10.0/task14.dll"

using task14;
using System;
using System.Diagnostics;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ScottPlot;

Func<double, double> sin = Math.Sin;
double a = -100;
double b = 100;
double exactResult = 0.0; 

double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
int[] threadCounts = { 1, 2, 4, 8, 16, 32 };
int iterations =30; 

double optimalStep = 0;
int optimalThreads = 1;
double bestMultiTime = double.MaxValue;
double bestSingleTime = 0;
double bestSpeedup = 0;

foreach (double step in steps)
{
    double result = DefiniteIntegral.Solve(a, b, sin, step, 4);
    double error = Math.Abs(result - exactResult);
    bool isAccurate = error < 1e-4;
    
    if (!isAccurate)
        continue;
    
    var stopwatch = new Stopwatch();
    double totalTimeSingle = 0;
    for (int i = 0; i < iterations; i++)
    {
        stopwatch.Restart();
        DefiniteIntegral.SingleThreadSolve(a, b, sin, step);
        stopwatch.Stop();
        totalTimeSingle += stopwatch.Elapsed.TotalMilliseconds;
    }
    double avgSingleTime = totalTimeSingle / iterations;
    
    double bestTimeForStep = double.MaxValue;
    int bestThreadsForStep = 1;
    
    foreach (int threads in threadCounts)
    {
        double totalTime = 0;
        for (int i = 0; i < iterations; i++)
        {
            stopwatch.Restart();
            DefiniteIntegral.Solve(a, b, sin, step, threads);
            stopwatch.Stop();
            totalTime += stopwatch.Elapsed.TotalMilliseconds;
        }
        double avgTime = totalTime / iterations;
        
        if (avgTime < bestTimeForStep)
        {
            bestTimeForStep = avgTime;
            bestThreadsForStep = threads;
        }
    }
    
    double speedupPercent = ((avgSingleTime - bestTimeForStep) / avgSingleTime) * 100;
    
    if (speedupPercent > 15.0)
    {
        optimalStep = step;
        optimalThreads = bestThreadsForStep;
        bestMultiTime = bestTimeForStep;
        bestSingleTime = avgSingleTime;
        bestSpeedup = speedupPercent;
        break;
    }
}

if (optimalStep == 0)
{
    optimalStep = steps.Last();
    optimalThreads = 4;
    bestSingleTime = 0;
    bestMultiTime = 0;
    bestSpeedup = 0;
}

var results = new List<(int threads, double time)>();
foreach (int threads in threadCounts)
{
    var stopwatch = new Stopwatch();
    double totalTime = 0;
    
    for (int i = 0; i < iterations; i++)
    {
        stopwatch.Restart();
        DefiniteIntegral.Solve(a, b, sin, optimalStep, threads);
        stopwatch.Stop();
        totalTime += stopwatch.Elapsed.TotalMilliseconds;
    }
    
    double averageTime = totalTime / iterations;
    results.Add((threads, averageTime));
}

var plt = new ScottPlot.Plot();
double[] plotTimes = results.Select(r => r.time).ToArray();
double[] plotThreads = results.Select(r => (double)r.threads).ToArray();
plt.Add.Scatter(plotTimes, plotThreads);
plt.Title("Performance");
plt.XLabel("time (ms)"); 
plt.YLabel("number of threads");
plt.SavePng("graph.png", 800, 600);

string report = $@"
отчет

1. оптимальный размер шага: {optimalStep} - шаг, обеспечивающий ошибку вычисления < 1e-4, при котором многопоточное решение выигрывает у более чем на 15%

2. оптимальное количество потоков: {optimalThreads}, при этом количестве потоков достигается мин. время выполнения Solve. Дальнейшее увеличение потоков не дает прироста

3. Сравнение с однопоточной версией:
   - Время однопоточной реализации: {bestSingleTime:F3} мс
   - Время многопоточной версии: {bestMultiTime:F3} мс
   - Разница: {bestSpeedup:F2}%
   многопоточная быстрее однопоточной на {bestSpeedup:F2}%
";
File.WriteAllText("report.txt", report);

Installed Packages ScottPlot, 5.1.59 SkiaSharp.NativeAssets.Linux.NoDependencies, 3.119.2